# 📚 Multi-PDF Chat Agent — 100% Free Version

A RAG (Retrieval-Augmented Generation) pipeline that lets you ask questions
across **multiple PDF documents** at once — no Google/OpenAI API key needed.

### What changed from the original code

| Original (Paid, needs `GOOGLE_API_KEY`) | This Notebook (Free) |
|---|---|
| `GoogleGenerativeAIEmbeddings` (Gemini) | `sentence-transformers` (`all-MiniLM-L6-v2`) |
| `ChatGoogleGenerativeAI` (`gemini-pro`) | `google/flan-t5-large` (local) |
| LangChain `FAISS.from_texts` / `load_local` | Raw `faiss` index, built and loaded manually |
| LangChain `load_qa_chain` | A simple custom prompt + `generate()` call |

### About ".h5" — why it isn't used here

`.h5` is a **Keras/TensorFlow weights format** for models you train yourself
(e.g. a CNN trained on MNIST). This project doesn't train anything — it
**loads a pre-trained LLM** (Flan-T5) from HuggingFace and uses it as-is,
in PyTorch format, not `.h5`.

The thing that actually gets "saved" in a RAG pipeline like this is the
**FAISS vector index** (the embedded PDF chunks) — that's the real
persisted artifact, and that's what we save to disk in Step 5 below.

### Pipeline
```
Multiple PDFs
      |
      v  pypdf
  Combined raw text
      |
      v  chunk_text()
  List of text chunks
      |
      v  sentence-transformers
  Chunk embeddings
      |
      v  faiss
  Vector index (saved to disk)
      |
      v  (at query time)
Question -> embedding -> similarity search -> top-k chunks
      |
      v  flan-t5-large
  Final answer
```


## Step 0 — Install Dependencies

In [1]:
%pip install pypdf sentence-transformers faiss-cpu transformers torch numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.9 MB/s eta 0:00:00:00:0100:01


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls

drive  sample_data


## Step 1 — Get Your PDF Files

You can work with **multiple PDFs at once** — that's the whole point of
this "Multi-PDF" chat agent.


In [5]:
# Option A: Upload directly in Colab (uncomment if using Google Colab)
# from google.colab import files
# uploaded = files.upload()
# pdf_paths = list(uploaded.keys())

# Option B: Use local file paths
pdf_paths = ["/content/drive/MyDrive/gate da 2026.pdf", "/content/drive/MyDrive/Mathematics_+_Reasoning_+_English EE 2024_copy.pdf"]   # <-- change these

print("PDFs to process:", pdf_paths)


PDFs to process: ['/content/drive/MyDrive/gate da 2026.pdf', '/content/drive/MyDrive/Mathematics_+_Reasoning_+_English EE 2024_copy.pdf']


## Step 2 — Extract Text from All PDFs

Reads every page of every PDF and concatenates all the text into one
big string — same as the original `get_pdf_text()` function.


In [6]:
from pypdf import PdfReader


def get_pdf_text(pdf_paths: list[str]) -> str:
    """Extract and concatenate text from multiple PDF files."""
    text = ""
    for pdf_path in pdf_paths:
        pdf_reader = PdfReader(pdf_path)
        for page in pdf_reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text
    return text


raw_text = get_pdf_text(pdf_paths)
print(f"Total characters extracted: {len(raw_text)}")
print("\n--- Preview ---\n")
print(raw_text[:500])


Total characters extracted: 11191

--- Preview ---

Q .1
O p tio n s A .
B .
C .
D .
Q u e s tio n  T y p e  : M C Q
Q u e s tio n  ID  : 2 2 8 4 8 2 1 0 0 1 4
S ta tu s  : A n s w e re d
C h o s e n  O p tio n  : C
G AT E  2 0 2 6  1 5 T H  F E B  2 0 2 6  S 2
C a n d id a te  ID D A 2 6 S 8 3 0 3 1 1 5 8
C a n d id a te  N a m e C H IR A G  V YA S
Te s t C e n te r N a m e S h iv  J y o ti In te rn a tio n a l S c h o o l
Te s t D a te 1 5 /0 2 /2 0 2 6
Te s t T im e 2 :3 0  P M  - 5 :3 0  P M
S u b je c t D A  D a ta  S c ie n c e  a n d  A r 


## Step 3 — Split Text into Chunks

Same purpose as `RecursiveCharacterTextSplitter` in the original code,
implemented with plain Python (no LangChain dependency needed).

- `chunk_size`: words per chunk
- `overlap`: words shared between consecutive chunks (preserves context
  across chunk boundaries)


In [7]:
def get_text_chunks(text: str, chunk_size: int = 1000, overlap: int = 150) -> list[str]:
    """Split text into overlapping word-based chunks."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


text_chunks = get_text_chunks(raw_text)
print(f"Total chunks created: {len(text_chunks)}")
print("\n--- First chunk preview ---\n")
print(text_chunks[0][:300])


Total chunks created: 6

--- First chunk preview ---

Q .1 O p tio n s A . B . C . D . Q u e s tio n T y p e : M C Q Q u e s tio n ID : 2 2 8 4 8 2 1 0 0 1 4 S ta tu s : A n s w e re d C h o s e n O p tio n : C G AT E 2 0 2 6 1 5 T H F E B 2 0 2 6 S 2 C a n d id a te ID D A 2 6 S 8 3 0 3 1 1 5 8 C a n d id a te N a m e C H IR A G V YA S Te s t C e n te


## Step 4 — Generate Embeddings (Free, Local)

Replaces `GoogleGenerativeAIEmbeddings(model="models/embedding-001")`
with `sentence-transformers` — runs entirely on your machine, no API
key, no cost.


In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(text_chunks, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

print("Embedding matrix shape:", chunk_embeddings.shape)
print("(num_chunks, embedding_dimension)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (6, 384)
(num_chunks, embedding_dimension)


## Step 5 — Build & Save the FAISS Vector Store

This is the **"save your model" step** for this project — instead of a
`.h5` file, the persisted artifact is a **FAISS index** (the vector
database of all your PDF chunks). This replaces
`vector_store.save_local("faiss_index")` from the original LangChain code.

We save two files:
- `faiss_index/index.faiss` — the actual vector index
- `faiss_index/chunks.txt` — the original text for each chunk (FAISS only
  stores numbers, not the source text, so we keep this alongside it)


In [9]:
import faiss
import os

FAISS_INDEX_PATH = "faiss_index"


def get_vector_store(text_chunks: list[str], embeddings: np.ndarray):
    """Build a FAISS index from embeddings and save it (+ chunk text) to disk."""
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    os.makedirs(FAISS_INDEX_PATH, exist_ok=True)
    faiss.write_index(index, os.path.join(FAISS_INDEX_PATH, "index.faiss"))

    with open(os.path.join(FAISS_INDEX_PATH, "chunks.txt"), "w", encoding="utf-8") as f:
        for chunk in text_chunks:
            f.write(chunk.replace("\n", " ") + "\n<<<CHUNK_END>>>\n")

    print(f"Saved FAISS index with {index.ntotal} vectors to '{FAISS_INDEX_PATH}/'")
    return index


index = get_vector_store(text_chunks, chunk_embeddings)


Saved FAISS index with 6 vectors to 'faiss_index/'


## Step 6 — Load the Saved Vector Store

Replaces `FAISS.load_local("faiss_index", embeddings)`. Demonstrates that
you can close this notebook, come back later, and reload the saved index
without re-processing the PDFs.


In [10]:
def load_vector_store():
    """Load a previously saved FAISS index and its chunk text from disk."""
    loaded_index = faiss.read_index(os.path.join(FAISS_INDEX_PATH, "index.faiss"))
    with open(os.path.join(FAISS_INDEX_PATH, "chunks.txt"), "r", encoding="utf-8") as f:
        raw = f.read()
    loaded_chunks = [c.strip() for c in raw.split("<<<CHUNK_END>>>") if c.strip()]
    return loaded_index, loaded_chunks


loaded_index, loaded_chunks = load_vector_store()
print(f"Loaded index with {loaded_index.ntotal} vectors and {len(loaded_chunks)} chunks")


Loaded index with 6 vectors and 6 chunks


## Step 7 — Retrieve Relevant Chunks

Replaces `new_db.similarity_search(user_question)`. Embeds the question
and finds the most similar chunks from the FAISS index.


In [11]:
def retrieve_relevant_chunks(query: str, chunks: list[str], index, top_k: int = 4) -> list[str]:
    """Return the top_k chunks most similar to the query."""
    query_embedding = embedding_model.encode([query]).astype("float32")
    _, indices = index.search(query_embedding, top_k)
    return [chunks[i] for i in indices[0] if i < len(chunks)]


# Quick test
test_query = "What is this document about?"
results = retrieve_relevant_chunks(test_query, loaded_chunks, loaded_index)

for i, r in enumerate(results, start=1):
    print(f"--- Chunk {i} ---")
    print(r[:250], "...\n")


--- Chunk 1 ---
s : A n s w e re d C h o s e n O p tio n : D Q .1 2 O p tio n s A . B . C . D . Q u e s tio n T y p e : M S Q Q u e s tio n ID : 2 2 8 4 8 2 1 0 0 3 5 S ta tu s : A n s w e re d C h o s e n O p tio n : B ,D Q .1 3 G iv e n A n s w e r : -- Q u e s ti ...

--- Chunk 2 ---
Q .1 O p tio n s A . B . C . D . Q u e s tio n T y p e : M C Q Q u e s tio n ID : 2 2 8 4 8 2 1 0 0 1 4 S ta tu s : A n s w e re d C h o s e n O p tio n : C G AT E 2 0 2 6 1 5 T H F E B 2 0 2 6 S 2 C a n d id a te ID D A 2 6 S 8 3 0 3 1 1 5 8 C a n d ...

--- Chunk 3 ---
6 S ta tu s : A n s w e re d C h o s e n O p tio n : B ,C Q .2 3 O p tio n s A . B . C . D . Q u e s tio n T y p e : M S Q Q u e s tio n ID : 2 2 8 4 8 2 1 0 0 3 4 S ta tu s : N o t A tte m p te d a n d M a rk e d F o r R e v ie w C h o s e n O p tio ...

--- Chunk 4 ---
s : A n s w e re d C h o s e n O p tio n : C Q .3 5 O p tio n s A . B . C . D . Q u e s tio n T y p e : M S Q Q u e s tio n ID : 2 2 8 4 8 2 1 0 0 6 4 S ta tu s : A n s

## Step 8 — Load a Free, Local LLM for Answer Generation

Replaces `ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.3)`.

We use `google/flan-t5-large` directly via `AutoTokenizer` +
`AutoModelForSeq2SeqLM` (not `pipeline()`, to avoid the
`KeyError: Unknown task` issue some transformers versions throw).


In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading Flan-T5-Large... (downloads ~800MB on first run)")
qa_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
qa_model     = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
print("Loaded successfully!")


Loading Flan-T5-Large... (downloads ~800MB on first run)


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded successfully!


## Step 9 — Build the QA Chain (Prompt + Generate)

Replaces `load_qa_chain(model, chain_type="stuff", prompt=prompt)`.
Uses the **exact same prompt template** as the original code.


In [13]:
def generate_answer(query: str, chunks: list[str], index, top_k: int = 4):
    """Run the full RAG pipeline: retrieve context, then generate an answer."""
    relevant_chunks = retrieve_relevant_chunks(query, chunks, index, top_k=top_k)
    context = "\n\n".join(relevant_chunks)

    prompt = f"""Answer the question as detailed as possible from the provided context,
make sure to provide all the details. If the answer is not in the provided
context, just say "answer is not available in the context". Don't provide
a wrong answer.

Context:
{context}

Question: {query}

Answer:"""

    inputs = qa_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = qa_model.generate(**inputs, max_length=300, num_beams=4, early_stopping=True)
    answer = qa_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return answer, relevant_chunks


## Step 10 — Ask Questions Across Your PDFs

In [14]:
question = "What is the main topic discussed across these documents?"
answer, sources = generate_answer(question, loaded_chunks, loaded_index)

print("Question:", question)
print("\nAnswer:", answer)

print("\n--- Source chunks used ---")
for i, s in enumerate(sources, start=1):
    print(f"\nSource {i}: {s[:200]}...")


Question: What is the main topic discussed across these documents?

Answer: Answer is not available in the context

--- Source chunks used ---

Source 1: Q .1 O p tio n s A . B . C . D . Q u e s tio n T y p e : M C Q Q u e s tio n ID : 2 2 8 4 8 2 1 0 0 1 4 S ta tu s : A n s w e re d C h o s e n O p tio n : C G AT E 2 0 2 6 1 5 T H F E B 2 0 2 6 S 2 C ...

Source 2: s : A n s w e re d C h o s e n O p tio n : D Q .1 2 O p tio n s A . B . C . D . Q u e s tio n T y p e : M S Q Q u e s tio n ID : 2 2 8 4 8 2 1 0 0 3 5 S ta tu s : A n s w e re d C h o s e n O p tio n ...

Source 3: 6 S ta tu s : A n s w e re d C h o s e n O p tio n : B ,C Q .2 3 O p tio n s A . B . C . D . Q u e s tio n T y p e : M S Q Q u e s tio n ID : 2 2 8 4 8 2 1 0 0 3 4 S ta tu s : N o t A tte m p te d a n...

Source 4: s : A n s w e re d C h o s e n O p tio n : C Q .3 5 O p tio n s A . B . C . D . Q u e s tio n T y p e : M S Q Q u e s tio n ID : 2 2 8 4 8 2 1 0 0 6 4 S ta tu s : A n s w e re d C h o s e n O p tio n .

In [15]:
# Try your own question
my_question = "Summarize the key points."
answer, sources = generate_answer(my_question, loaded_chunks, loaded_index)

print("Question:", my_question)
print("\nAnswer:", answer)


Question: Summarize the key points.

Answer: answer is not available in the context


## Summary

| Step | Replaces | Free Tool Used |
|---|---|---|
| Text extraction | `PyPDF2` | `pypdf` (actively maintained fork) |
| Chunking | `RecursiveCharacterTextSplitter` | Custom `get_text_chunks()` |
| Embeddings | `GoogleGenerativeAIEmbeddings` | `sentence-transformers` |
| Vector store | LangChain `FAISS` wrapper | Raw `faiss` (saved/loaded manually) |
| LLM | `ChatGoogleGenerativeAI` (gemini-pro) | `google/flan-t5-large` |
| QA chain | `load_qa_chain` | Custom prompt + `generate()` |

**No `GOOGLE_API_KEY` or any other API key is required anywhere in this notebook.**

## Next Step

This exact pipeline is wrapped in a Streamlit web UI in `app.py`, supporting
multi-file upload directly in the browser. Run it with:
```bash
streamlit run app.py
```
